# nb_04_queries: the six business questions, in Spark SQL against Gold
**Task 4.** Each query returns exactly the columns named in the assignment, with the stated rounding.

## (a) Which machine types tie up the longest? (returned rentals only)

In [ ]:
qa = spark.sql("""
SELECT asset_type,
       ROUND(AVG(rental_duration_days), 2) AS avg_duration_days,
       COUNT(*)                            AS rentals
FROM gold_fact_rental
WHERE is_returned = 1
GROUP BY asset_type
ORDER BY avg_duration_days DESC
""")
display(qa)

## (b) Which depots are running hot? Fleet utilisation

In [ ]:
qb = spark.sql("""
SELECT d.depot_name,
       d.fleet_size,
       ROUND(SUM(f.asset_days_in_window), 1)                              AS asset_days_used,
       ROUND(SUM(f.asset_days_in_window) * 100.0 / (d.fleet_size * 30), 1) AS utilisation_pct
FROM gold_fact_rental f
JOIN gold_dim_depot  d ON f.depot_code = d.depot_code
GROUP BY d.depot_name, d.fleet_size
ORDER BY utilisation_pct DESC
""")
display(qb)

## (c) Revenue by payer type

In [ ]:
qc = spark.sql("""
SELECT payer_type,
       COUNT(*)                    AS bills,
       ROUND(SUM(amount_inr), 0)   AS revenue,
       ROUND(AVG(amount_inr), 0)   AS avg_bill
FROM gold_fact_billing
GROUP BY payer_type
ORDER BY revenue DESC
""")
display(qc)

## (d) Revenue by depot: billing joined to rentals joined to the depot dimension

In [ ]:
qd = spark.sql("""
SELECT d.depot_name,
       COUNT(b.bill_id)                                          AS bills,
       ROUND(SUM(b.amount_inr), 0)                               AS revenue,
       ROUND(SUM(b.amount_inr) / COUNT(DISTINCT r.rental_id), 0) AS revenue_per_rental
FROM gold_fact_billing b
JOIN gold_fact_rental  r ON b.rental_id = r.rental_id
JOIN gold_dim_depot    d ON r.depot_code = d.depot_code
GROUP BY d.depot_name
ORDER BY revenue DESC
""")
display(qd)

## (e) Priority mix: share of rentals by rental type

In [ ]:
qe = spark.sql("""
SELECT rental_type,
       COUNT(*)                                          AS rentals,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM gold_fact_rental
GROUP BY rental_type
ORDER BY rentals DESC
""")
display(qe)

## (f) What is out right now? Machines currently out by depot

In [ ]:
qf = spark.sql("""
SELECT d.depot_name,
       COUNT(*) AS currently_out
FROM gold_fact_rental r
JOIN gold_dim_depot  d ON r.depot_code = d.depot_code
WHERE r.is_returned = 0
GROUP BY d.depot_name
ORDER BY currently_out DESC
""")
display(qf)

## Reconciliation checks (screenshot this cell's output)

In [ ]:
from pyspark.sql import functions as F

total_rev = float(qd.agg(F.sum("revenue")).first()[0])
print(f"(d) total revenue across depots = Rs {total_rev:,.0f}  (about {total_rev/1e7:.2f} crore)")
assert 4.5e7 <= total_rev <= 5.5e7, "expected roughly five crore"

out_gold   = qf.agg(F.sum("currently_out")).first()[0]
out_silver = spark.table("silver_rentals").filter(F.col("checkin_ts").isNull()).count()
print(f"(f) currently out: Gold = {out_gold}, Silver = {out_silver}")
assert out_gold == out_silver == 175

print("(c) bills total =", qc.agg(F.sum("bills")).first()[0], "(expect 779)")
print("(e) pct total   =", round(qe.agg(F.sum("pct")).first()[0], 1), "(expect about 100)")
print("Top depot by revenue :", qd.orderBy(F.desc("revenue")).first()["depot_name"])
print("Top depot per rental :", qd.orderBy(F.desc("revenue_per_rental")).first()["depot_name"])